# HopeGait — Colab LOSO Runner (GPU)

Full Brain-phase run on a Colab GPU (Stanford NMBL only):
**EDA → baseline → LOSO training → evaluation → (optional) int8 edge delta**.

Survives Colab session limits: training checkpoints **per fold**, so re-running
the training cell skips folds already done. Keep `models/` and `data/` on Drive.

> **Prerequisite:** `data/` is gitignored, so `git pull` does NOT bring it.
> Upload your local `data/` (raw or already-preprocessed `data/processed/`)
> to the Drive working dir below before running §4+.

> Runtime → Change runtime type → **T4 GPU** first.

## 0. GPU check

In [ ]:
!nvidia-smi

## 1. Mount Drive
Repo + dataset live here so models/reports persist across sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/hopegait'
import os; os.makedirs(WORK, exist_ok=True)
%cd $WORK

## 2. Clone (first session only)

In [ ]:
import os
REPO = 'https://github.com/truong-tt/HopeGait-FoG_Detection_via_IMU_and_ML.git'
if not os.path.exists('HopeGait-FoG_Detection_via_IMU_and_ML'):
    !git clone $REPO
%cd HopeGait-FoG_Detection_via_IMU_and_ML
!git pull --ff-only

## 3. Install training deps
Colab ships a CUDA torch + numpy 2.x; `requirements.txt` no longer caps numpy,
so this won't downgrade it. If an earlier run broke numpy, this cell restarts
the runtime once — re-run it after the restart.

In [ ]:
!pip -q install -r requirements.txt
import os
os.environ['PYTHONPATH'] = os.path.abspath('src')
try:
    import torch, numpy, scipy.signal  # noqa: F401
    print('torch', torch.__version__, 'cuda', torch.cuda.is_available(),
          '| numpy', numpy.__version__, '| PYTHONPATH', os.environ['PYTHONPATH'])
except Exception as e:
    print('Broken env — restarting, then re-run this cell:', e)
    os.kill(os.getpid(), 9)

## 4. Dataset
Upload your local `data/` here first. If you already have `data/processed/`,
skip preprocessing. Otherwise place raw CSV/XLSX under `data/raw/` and run it.

In [ ]:
import glob
print('processed windows:', len(glob.glob('data/processed/win_128/*_x.npy')))
if not glob.glob('data/processed/win_128/*_x.npy'):
    !python src/data_pipeline/preprocess.py

## 5. EDA + non-deep baseline (problem analysis)

In [ ]:
!python scripts/eda.py
!python src/baselines/freeze_index_baseline.py

## 6. LOSO training (GPU, resumable)
Per-fold checkpointing → re-run after a session drop to continue.

**Hyperparameters:** EMA decay **0.99** (not 0.999 — the default tracks too
slowly on this small 7-subject set, leaving the saved EMA weights near init)
and **60 epochs** so folds converge. AMP on for GPU speed.

In [ ]:
import os, sys, subprocess
sys.path.insert(0, 'src')
from data_pipeline.dataset import get_all_subjects
WIN, EPOCHS = 128, 60
os.environ['HOPEGAIT_EMA_DECAY'] = '0.99'   # faster EMA for small data
data_dir = f'data/processed/win_{WIN}'
subjects = get_all_subjects(data_dir)
print(f'{len(subjects)} subjects:', subjects)
for subj in subjects:
    ckpt = f'models/win_{WIN}/hopegait_tcn_best_subj{subj}.pth'
    if os.path.exists(ckpt):
        print('skip (done):', subj); continue
    print('=== training fold', subj, '===')
    subprocess.run([sys.executable, 'src/training/train.py',
                    '--subject', str(subj), '--window', str(WIN),
                    '--epochs', str(EPOCHS), '--use-amp'], check=True)

## 7. Evaluate (per-subject table + mean±std)

In [ ]:
!python src/training/evaluate.py
from IPython.display import Markdown, display
for f in ('reports/results_table.md', 'reports/eda_summary.md'):
    if os.path.exists(f):
        display(Markdown(open(f).read()))

## 8. (Optional) Edge int8 delta
TF 2.15 needs numpy<2 — run this in a **fresh runtime** (Runtime → Disconnect
and delete runtime, reconnect, run §1–4, then this).

In [ ]:
SUBJECT = '3'   # a subject whose fold you trained
!pip -q install -r requirements-edge.txt
!python src/edge_conversion/quantize_model.py --subject $SUBJECT --window 128
!python src/edge_conversion/evaluate_tflite.py --subject $SUBJECT --window 128
import json
print(json.dumps(json.load(open('models/int8_delta.json')), indent=2))